In [3]:
import pandas as pd
import numpy as np

In [10]:
# 1. Leer el CSV
df_votaciones = pd.read_csv('hcdn_votaciones_historico.csv')

# 2. Eliminar las columnas inútiles
df_votaciones = df_votaciones.drop(columns=['Unnamed: 0', '¿QUÉ DIJO?'])

# 3. Renombrar las columnas para no tener mayúsculas, espacios ni signos de interrogación
df_votaciones = df_votaciones.rename(columns={
    'DIPUTADO': 'diputado',
    'BLOQUE': 'bloque',
    'PROVINCIA': 'provincia',
    '¿CÓMO VOTÓ?': 'voto'
})

# 4. Convertir la fecha (que ahora es texto) a un formato de Fecha real (Datetime)
# Esto es vital para después poder filtrar por "Año" o "Mes" en el dashboard
df_votaciones['fecha_votacion'] = pd.to_datetime(df_votaciones['fecha_votacion'], format='%d/%m/%Y', errors='coerce')

# Vemos el resultado limpio
print(df_votaciones.head())
print("\nEstructura final del dataset:")
print(df_votaciones.info())

votaciones.head(8)

                            diputado                        bloque  \
0  ABDALA DE MATARAZZO, Norma Amanda    Frente Cívico por Santiago   
1                 ABRAHAM, Alejandro  Frente para la Victoria - PJ   
2                  AGUAD, Oscar Raúl          Unión Cívica Radical   
3               AGUILAR, Lino Walter            Compromiso Federal   
4             ALEGRE, Gilberto Oscar              Frente Renovador   

             provincia        voto  id_votacion  \
0  Santiago del Estero  AFIRMATIVO            1   
1              Mendoza  AFIRMATIVO            1   
2              Córdoba  AFIRMATIVO            1   
3             San Luis     AUSENTE            1   
4         Buenos Aires  AFIRMATIVO            1   

                                     titulo_proyecto fecha_votacion  
0  Régimen previsional especial de carácter excep...     2015-10-07  
1  Régimen previsional especial de carácter excep...     2015-10-07  
2  Régimen previsional especial de carácter excep...     2015-

,diputado,bloque,provincia,voto,id_votacion,titulo_proyecto,fecha_votacion
0,"ABDALA DE MATARAZZO, Norma Amanda",Frente Cívico por Santiago,Santiago del Estero,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07
1,"ABRAHAM, Alejandro",Frente para la Victoria - PJ,Mendoza,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07
2,"AGUAD, Oscar Raúl",Unión Cívica Radical,Córdoba,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07
3,"AGUILAR, Lino Walter",Compromiso Federal,San Luis,AUSENTE,1,Régimen previsional especial de carácter excep...,2015-10-07
4,"ALEGRE, Gilberto Oscar",Frente Renovador,Buenos Aires,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07
5,"ALFONSÍN, Ricardo",Unión Cívica Radical,Buenos Aires,AUSENTE,1,Régimen previsional especial de carácter excep...,2015-10-07
6,"ALONSO, Laura",Unión PRO,C.A.B.A.,AUSENTE,1,Régimen previsional especial de carácter excep...,2015-10-07
7,"ALONSO, María Luz",Frente para la Victoria - PJ,La Pampa,AFIRMATIVO,1,Régimen previsional especial de carácter excep...,2015-10-07


In [12]:
import unicodedata
import re

# ── 1. Función de normalización ───────────────────────────────────────────────
def normalizar_nombre(s):
    """Quita tildes, pasa a mayúsculas, elimina caracteres no alfabéticos."""
    if not isinstance(s, str):
        return ''
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = s.upper().strip()
    s = re.sub(r'[^A-Z, ]', '', s)
    s = re.sub(r'\s+', ' ', s)
    return s

# ── 2. Cargar nómina actual y construir el set de nombres normalizados ─────────
df_actuales = pd.read_csv('diputados_actuales.csv')

df_actuales['nombre_norm'] = (
    df_actuales['Apellido'].fillna('') + ', ' + df_actuales['Nombre'].fillna('')
).apply(normalizar_nombre)

nombres_actuales = set(df_actuales['nombre_norm'])

# ── 3. Normalizar columna diputado en df_consolidado ──────────────────────────
df_votaciones['diputado_norm'] = df_votaciones['diputado'].apply(normalizar_nombre)

# ── 4. Filtrar ────────────────────────────────────────────────────────────────
df_modelo = df_votaciones[df_votaciones['diputado_norm'].isin(nombres_actuales)].copy()

# ── 5. Diagnóstico ────────────────────────────────────────────────────────────
print(f"Registros originales:  {len(df_votaciones):,}")
print(f"Registros filtrados:   {len(df_modelo):,}")
print(f"Reducción:             {(1 - len(df_modelo)/len(df_votaciones))*100:.1f}%")
print(f"Diputados únicos en df_modelo: {df_modelo['diputado_norm'].nunique()}")

Registros originales:  578,507
Registros filtrados:   88,083
Reducción:             84.8%
Diputados únicos en df_modelo: 257


In [14]:
import re
import pandas as pd

# ============================================================
# PASO 1 — Eliminar ruido procedimental
# ============================================================
patrones_ruido = [
    r'APARTAMIENTO DE REGLAMENTO',
    r'Apartamiento de [Rr]eglamento',
    r'Apartamiento del [Rr]eglamento',
    r'A N U L A D A',
    r'Plan de Labor Parlamentaria',
    r'Conjunto de proyectos',
    r'Conjunto de varios',
    r'^\s*-\s*Votación',
    r'MOCIÓN SOLICITADA POR',
    r'Moción solicitada por',
]
regex_ruido = '|'.join(patrones_ruido)
df = df_modelo[~df_modelo['titulo_proyecto'].str.contains(
    regex_ruido, regex=True, na=False
)].copy()
print(f"[1] Registros originales:          {len(df_modelo):,}")
print(f"[1] Registros tras eliminar ruido: {len(df):,}")
print(f"[1] Ruido eliminado:               {len(df_modelo)-len(df):,}")


[1] Registros originales:          88,083
[1] Registros tras eliminar ruido: 77,284
[1] Ruido eliminado:               10,799
